# Count Downloaded Non-Speech Audio Clips

This notebook answers one question:

> How many clips labeled **non_speech** in the labels CSV also exist as downloaded files in `audio_cache/`?

**How matching works**
1. Read labels from `audio_speech_labels.csv` (project root)
2. Keep rows where `speech_label == "non_speech"`
3. Collect downloaded file IDs from `audio_cache/` (filenames look like `{id}.mp3`)
4. Intersect the two sets and print the count

This notebook lives in `Jupyter Notebook/`, so paths resolve relative to the project root one level up.

## 1. Setup

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
# This notebook is in Jupyter Notebook/; data lives one level up in the project root.
PROJECT_ROOT = Path("..").resolve()

CSV_PATH = PROJECT_ROOT / "audio_speech_labels.csv"
AUDIO_CACHE_DIR = PROJECT_ROOT / "audio_cache"

ID_COLUMN = "id"
LABEL_COLUMN = "speech_label"
NON_SPEECH_VALUE = "non_speech"

print(f"Project root: {PROJECT_ROOT}")
print(f"Labels CSV:   {CSV_PATH}")
print(f"Audio cache:  {AUDIO_CACHE_DIR}")

## 2. Helper functions

In [ ]:
def load_labels(csv_path: Path) -> pd.DataFrame:
    """Load the speech labels CSV."""
    if not csv_path.exists():
        raise FileNotFoundError(f"Labels file not found: {csv_path}")
    return pd.read_csv(csv_path)


def filter_non_speech(
    labels: pd.DataFrame,
    label_column: str = LABEL_COLUMN,
    non_speech_value: str = NON_SPEECH_VALUE,
) -> pd.DataFrame:
    """Return only rows labeled as non-speech."""
    return labels[labels[label_column] == non_speech_value].copy()


def get_downloaded_ids(cache_dir: Path) -> set[str]:
    """Collect audio IDs from filenames in the cache folder.

    Skips incomplete downloads ending in `.partial`.
    Example: `abc-123.mp3` → id `abc-123`
    """
    if not cache_dir.exists():
        raise FileNotFoundError(f"Audio cache not found: {cache_dir}")

    return {
        path.stem
        for path in cache_dir.iterdir()
        if path.is_file() and not path.name.endswith(".partial")
    }


def match_non_speech_with_downloads(
    non_speech: pd.DataFrame,
    downloaded_ids: set[str],
    id_column: str = ID_COLUMN,
) -> pd.DataFrame:
    """Keep non-speech rows whose id has a downloaded audio file."""
    return non_speech[
        non_speech[id_column].astype(str).isin(downloaded_ids)
    ].copy()

## 3. Load labels and keep non-speech rows

In [ ]:
labels = load_labels(CSV_PATH)
non_speech = filter_non_speech(labels)

print(f"Total rows in CSV:     {len(labels)}")
print(f"Non-speech labeled:    {len(non_speech)}")

## 4. Collect downloaded file IDs

In [ ]:
downloaded_ids = get_downloaded_ids(AUDIO_CACHE_DIR)

print(f"Downloaded files:      {len(downloaded_ids)}")

## 5. Match and count

In [ ]:
matched = match_non_speech_with_downloads(non_speech, downloaded_ids)

print(f"Non-speech with downloaded file: {len(matched)}")